# 01 — Data Understanding & Exploratory Data Analysis

First pass at understanding the two raw datasets before any cleaning, transformation, or analysis begins.

**Ground rules for this notebook:**
- No data cleaning
- No visualizations
- No conclusions
- The property tax file (~423 MB) is sampled at 1,000 rows only

---

## 1. Project Context

This project examines the relationship between **housing supply** and **assessed property values** in the City of Vancouver.

We are working with two raw datasets:

| Dataset | File | Role |
|---|---|---|
| Property Tax Report | `property_tax_report_raw.csv` | Assessed land and improvement values |
| Issued Building Permits | `issued_building_permits_raw.csv` | Permit activity as a housing supply signal |

**Why sample first?**  
The property tax file is approximately 423 MB. Loading it in full during exploration would be slow and memory-intensive. We load 1,000 rows to understand structure and data quality, then scale up only when needed.

## 2. Import Libraries

We use only the Python standard library (`os`, `csv`) and `pandas`. No additional packages are needed at this stage.

In [1]:
import os       # file path operations and existence checks
import csv      # delimiter sniffing without loading the full file
import pandas as pd  # tabular data loading and inspection

# Show up to 50 columns in notebook output (default pandas limit is 20)
pd.set_option('display.max_columns', 50)

print('pandas version:', pd.__version__)

pandas version: 3.0.3


> **What to check:** `pandas` should import without errors. If you see a `ModuleNotFoundError`, activate your virtual environment and run `pip install -r requirements.txt` from the project root, then restart the kernel.

## 3. Define File Paths

All raw files live in `data/raw/`. Paths are defined once here so they are easy to update if files move. We also verify that both files exist and display their sizes before attempting to load anything.

In [2]:
# Navigate up one level from notebooks/ to the project root
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))

PROPERTY_TAX_PATH = os.path.join(BASE_DIR, 'data', 'raw', 'property_tax_report_raw.csv')
PERMITS_PATH      = os.path.join(BASE_DIR, 'data', 'raw', 'issued_building_permits_raw.csv')

# Confirm both files exist before proceeding
for label, path in [('Property Tax', PROPERTY_TAX_PATH), ('Permits', PERMITS_PATH)]:
    exists = os.path.isfile(path)
    size   = os.path.getsize(path) if exists else 0
    status = 'FOUND' if exists else 'MISSING'
    print(f'{label:15s} : {status} — {size:,} bytes')
    print(f'               {path}')

Property Tax    : FOUND — 443,785,841 bytes
               c:\Users\User\Documents\vancouver-property-value-housing-supply-analysis\data\raw\property_tax_report_raw.csv
Permits         : FOUND — 56,671,596 bytes
               c:\Users\User\Documents\vancouver-property-value-housing-supply-analysis\data\raw\issued_building_permits_raw.csv


> **What to check:** Both files should show `FOUND`. Expected sizes: ~443,785,841 bytes for the property tax file and ~56,671,596 bytes for the permits file. If a file shows `MISSING`, confirm it was downloaded and placed in `data/raw/` with the exact filenames defined above.

## 4. Detect CSV Delimiter

Most CSV files use a comma (`,`), but some use semicolons (`;`), tabs (`\t`), or pipes (`|`). Python's built-in `csv.Sniffer` reads a small chunk of the file and infers the delimiter automatically.

Detecting this upfront prevents a common silent failure where every row loads as a single column because the wrong separator was assumed.

In [3]:
def detect_delimiter(filepath, num_bytes=8192):
    '''
    Read a small chunk of the file and sniff its CSV delimiter.
    Falls back to comma if sniffing is inconclusive.
    '''
    with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
        sample = f.read(num_bytes)
    try:
        dialect = csv.Sniffer().sniff(sample, delimiters=',;\t|')
        return dialect.delimiter
    except csv.Error:
        return ','  # safe default

prop_delim    = detect_delimiter(PROPERTY_TAX_PATH)
permits_delim = detect_delimiter(PERMITS_PATH)

print(f'Property Tax delimiter : {repr(prop_delim)}')
print(f'Permits delimiter      : {repr(permits_delim)}')

Property Tax delimiter : ';'
Permits delimiter      : ';'


> **What to check:** Both should return `','`. If you see `';'` or `'\t'`, the file uses a non-standard separator — the loader below will still work correctly since it uses the detected delimiter. If the output shows `','` but data later appears in a single column, open the file in a text editor to verify and override `sep` manually.

## 5. Load a Small Sample (1,000 Rows)

We load the first 1,000 rows of each file using `nrows=1000`. This is enough to inspect structure and data quality without reading hundreds of megabytes into memory.

`low_memory=False` tells pandas to infer column types using the full sample rather than guessing in chunks, which avoids mixed-type warnings on large files.

In [4]:
# Load first 1,000 rows only — do not remove the nrows limit during EDA
prop_sample = pd.read_csv(
    PROPERTY_TAX_PATH,
    nrows=1000,
    sep=prop_delim,
    low_memory=False,
    encoding='utf-8',
    encoding_errors='replace',  # replace unreadable characters instead of raising an error
)

permits_sample = pd.read_csv(
    PERMITS_PATH,
    nrows=1000,
    sep=permits_delim,
    low_memory=False,
    encoding='utf-8',
    encoding_errors='replace',
)

print('Samples loaded successfully.')

Samples loaded successfully.


> **What to check:** You should see `Samples loaded successfully.` If you get a `FileNotFoundError`, re-run the file path cell above. A persistent `UnicodeDecodeError` can be resolved by switching to `encoding='latin-1'`. If a file loads but all data appears in one column, the sniffed delimiter was incorrect — override `sep` with the correct character.

## 6. Shape of Each Sample

`.shape` returns `(rows, columns)`. Since we set `nrows=1000`, the row count will be 1,000 (unless the file has fewer rows). The **column count** is what tells us whether the delimiter was parsed correctly.

In [5]:
print(f'Property Tax sample shape  : {prop_sample.shape}   → (rows, columns)')
print(f'Permits sample shape       : {permits_sample.shape}   → (rows, columns)')

Property Tax sample shape  : (1000, 30)   → (rows, columns)
Permits sample shape       : (1000, 20)   → (rows, columns)


> **What to check:** The column count should be greater than 3–4. A result of `(1000, 1)` means the file was not parsed correctly — all data ended up in a single column, which usually means the wrong delimiter was detected.

## 7. Column Names

Listing all column names gives us a map of every available field. We will use this to populate `docs/data_dictionary.md` and decide which columns are relevant to the analysis.

In [6]:
print('=== Property Tax Columns ===')
for i, col in enumerate(prop_sample.columns):
    print(f'  [{i:02d}] {col}')

print()

print('=== Permits Columns ===')
for i, col in enumerate(permits_sample.columns):
    print(f'  [{i:02d}] {col}')

=== Property Tax Columns ===
  [00] PID
  [01] LEGAL_TYPE
  [02] FOLIO
  [03] LAND_COORDINATE
  [04] ZONING_DISTRICT
  [05] ZONING_CLASSIFICATION
  [06] LOT
  [07] PLAN
  [08] BLOCK
  [09] DISTRICT_LOT
  [10] FROM_CIVIC_NUMBER
  [11] TO_CIVIC_NUMBER
  [12] STREET_NAME
  [13] PROPERTY_POSTAL_CODE
  [14] NARRATIVE_LEGAL_LINE1
  [15] NARRATIVE_LEGAL_LINE2
  [16] NARRATIVE_LEGAL_LINE3
  [17] NARRATIVE_LEGAL_LINE4
  [18] NARRATIVE_LEGAL_LINE5
  [19] CURRENT_LAND_VALUE
  [20] CURRENT_IMPROVEMENT_VALUE
  [21] TAX_ASSESSMENT_YEAR
  [22] PREVIOUS_LAND_VALUE
  [23] PREVIOUS_IMPROVEMENT_VALUE
  [24] YEAR_BUILT
  [25] BIG_IMPROVEMENT_YEAR
  [26] TAX_LEVY
  [27] NEIGHBOURHOOD_CODE
  [28] REPORT_YEAR
  [29] NOTE

=== Permits Columns ===
  [00] PermitNumber
  [01] PermitNumberCreatedDate
  [02] IssueDate
  [03] PermitElapsedDays
  [04] ProjectValue
  [05] TypeOfWork
  [06] Address
  [07] ProjectDescription
  [08] PermitCategory
  [09] Applicant
  [10] ApplicantAddress
  [11] PropertyUse
  [12] Specif

> **What to look for:**
> - **Property Tax:** fields for assessed value, land value, improvement value, a tax or assessment year, and a property identifier (parcel ID or PID). Also note any address, neighbourhood, or zoning fields that could support geographic grouping.
> - **Permits:** fields for permit type, issue date, project value, address, and type of work (new construction vs. renovation vs. demolition). A date field and a type/category field are both essential for the time-trend analysis.
> - Flag any column names that are unclear, unexpectedly absent, or appear duplicated — note them for follow-up.

## 8. First 5 Rows

Viewing actual data values helps spot formatting issues before any analysis: currency symbols embedded in numeric fields, inconsistent date formats, extra whitespace, or placeholder values standing in for nulls.

In [7]:
print('=== Property Tax — First 5 Rows ===')
display(prop_sample.head())

print()
print('=== Permits — First 5 Rows ===')
display(permits_sample.head())

=== Property Tax — First 5 Rows ===


,PID,LEGAL_TYPE,FOLIO,LAND_COORDINATE,ZONING_DISTRICT,ZONING_CLASSIFICATION,LOT,PLAN,BLOCK,DISTRICT_LOT,FROM_CIVIC_NUMBER,TO_CIVIC_NUMBER,STREET_NAME,PROPERTY_POSTAL_CODE,NARRATIVE_LEGAL_LINE1,NARRATIVE_LEGAL_LINE2,NARRATIVE_LEGAL_LINE3,NARRATIVE_LEGAL_LINE4,NARRATIVE_LEGAL_LINE5,CURRENT_LAND_VALUE,CURRENT_IMPROVEMENT_VALUE,TAX_ASSESSMENT_YEAR,PREVIOUS_LAND_VALUE,PREVIOUS_IMPROVEMENT_VALUE,YEAR_BUILT,BIG_IMPROVEMENT_YEAR,TAX_LEVY,NEIGHBOURHOOD_CODE,REPORT_YEAR,NOTE
0,013-172-549,LAND,830212830000,83021283,R1-1,Residential Inclusive,4,VAP3089,A,327,NaN,961.0,MARINE DR SE,V5X 2V2,"LOT 4, BLOCK A, PLAN VAP3089, DISTR","ICT LOT 327, GROUP 1, NEW WESTMINST","ER LAND DISTRICT, EXC PT IN EXPL PL","6777, OF LOTS 5 TO 8",NaN,1250000.0,767000.0,2023.0,1161192.0,799000.0,2017.0,2017.0,6678.66,17,2023,NaN
1,030-022-843,STRATA,320718950602,32071895,CD-1 (545),Comprehensive Development,602,EPS3434,6,36,3203,5665.0,BOUNDARY RD,V5R 0E4,LOT 602 BLOCK 6 PLAN EPS3434 DIS,"TRICT LOT 36 NWD GROUP 1, TOGETHER",WITH AN INTEREST IN THE COMMON PRO,PERTY IN PROPORTION TO THE UNIT ENT,ITLEMENT OF THE STRATA LOT AS SHOWN,566000.0,322000.0,2023.0,507000.0,306000.0,2016.0,2016.0,2469.26,23,2023,NaN
2,027-537-978,STRATA,592118060076,59211806,DD,Comprehensive Development,76,BCS2936,NaN,185,2504,1188.0,PENDER ST W,V6E 0A2,LOT 76 PLAN BCS2936 DISTRICT LOT,185 NWD GROUP 1.,NaN,NaN,NaN,793000.0,427000.0,2023.0,800000.0,415000.0,2008.0,2008.0,3392.44,26,2023,NaN
3,028-763-017,STRATA,670026450025,67002645,C-2,Commercial,25,BCS4338,148,540,401,4355.0,10TH AVE W,V6R 2H6,LOT 25 BLOCK 148 PLAN BCS4338 DI,"STRICT LOT 540 NWD GROUP 1, TOGETH",ER WITH AN INTEREST IN THE COMMON P,ROPERTY IN PROPORTION TO THE UNIT E,NTITLEMENT OF THE STRATA LOT AS SHO,1157477.0,347000.0,2023.0,1009000.0,311000.0,2012.0,2012.0,4183.51,1,2023,NaN
4,028-763-050,STRATA,670026450029,67002645,C-2,Commercial,29,BCS4338,148,540,405,4355.0,10TH AVE W,V6R 2H6,LOT 29 BLOCK 148 PLAN BCS4338 DI,"STRICT LOT 540 NWD GROUP 1, TOGETH",ER WITH AN INTEREST IN THE COMMON P,ROPERTY IN PROPORTION TO THE UNIT E,NTITLEMENT OF THE STRATA LOT AS SHO,493000.0,182000.0,2023.0,427000.0,175000.0,2012.0,2012.0,1876.98,1,2023,NaN



=== Permits — First 5 Rows ===


,PermitNumber,PermitNumberCreatedDate,IssueDate,PermitElapsedDays,ProjectValue,TypeOfWork,Address,ProjectDescription,PermitCategory,Applicant,ApplicantAddress,PropertyUse,SpecificUseCategory,BuildingContractor,BuildingContractorAddress,IssueYear,GeoLocalArea,Geom,YearMonth,geo_point_2d
0,BP-2025-04512,2025-10-21,2025-10-28,7,190000.0,Addition / Alteration,"5918 ADERA STREET, Vancouver, BC V6M 3J4",Direct to Inspections - Addition / Alteration ...,Renovation - Residential - Lower Complexity,Jonathan Katz DBA: J & R Katz Design + Archite...,"4545 Langara Avenue\r\nVancouver, BC V6R 1C9",Dwelling Uses,Single Detached House,NaN,NaN,2025,Kerrisdale,"{""coordinates"": [-123.1428549, 49.2323258], ""t...",2025-10,"49.2323258, -123.1428549"
1,DB-2025-04513,2025-10-21,2025-11-10,20,468000.0,Addition / Alteration,"1408 ROBSON STREET, Vancouver, BC",Field Review - Addition / Alteration - 3401-14...,Renovation - Residential - Lower Complexity,Gloria Song,"Suite720\r\n999 West Broadway\r\nVancouver, BC...",Dwelling Uses,Apartment,NaN,NaN,2025,West End,"{""coordinates"": [-123.130607, 49.2882932], ""ty...",2025-11,"49.2882932, -123.130607"
2,BP-2025-04520,2025-10-22,2026-03-04,133,100000.0,Addition / Alteration,"1077 MARINASIDE CRESCENT #1106, Vancouver, BC ...",Direct to Inspections - Addition / Alteration ...,Renovation - Residential - Lower Complexity,Michael Elliston DBA: Hammerhead Consulting Inc.,"2023 W 36th Ave \r\nVancouver, BC V6M 1L1",Dwelling Uses,Multiple Dwelling,Hammerhead Consulting Inc,"2027 W 36TH AV \r\nVancouver, BC V6M 1L1",2026,Downtown,"{""coordinates"": [-123.1181418, 49.2739315], ""t...",2026-03,"49.2739315, -123.1181418"
3,BP-2025-04525,2025-10-22,2026-03-25,154,980000.0,New Building,"2541 E 2ND AVENUE, Vancouver, BC V5M 1C7",Low Density Housing - New Building - To constr...,New Build - Low Density Housing,Harjit Deol DBA: Apple Construction Ltd.,"13564 67 Ave\r\nSurrey, BC V3W2B8",Dwelling Uses,Multiple Dwelling,Stat Construction Ltd.,NaN,2026,Hastings-Sunrise,"{""coordinates"": [-123.0536547, 49.2687405], ""t...",2026-03,"49.2687405, -123.0536547"
4,BP-2025-04545,2025-10-23,2026-03-05,133,1000000.0,Addition / Alteration,"650 SW MARINE DRIVE, Vancouver, BC",Certified Professional Program - Addition / Al...,NaN,Kai Mikkelsen DBA: Thorson Consulting CP,2015 Main Street\r\nThorson Consulting Ltd.\r\...,Service Uses,Restaurant - Class 1,NaN,NaN,2026,Marpole,"{""coordinates"": [-123.120391, 49.2087022], ""ty...",2026-03,"49.2087022, -123.120391"


> **What to look for:**
> - Are value/amount columns stored as numbers, or as strings containing `$` or `,`?
> - Are date columns formatted consistently (e.g., `YYYY-MM-DD`, `DD/MM/YYYY`, `Month DD, YYYY`)?
> - Do any fields contain free-text or mixed content that will need parsing later?
> - Are there visible placeholder values — `0`, `N/A`, `UNKNOWN`, `-`, or empty strings — that may represent missing data but won't be caught by `isnull()`?

In [8]:
print("PERMITS COLUMNS:")
print(list(permits_sample.columns))

PERMITS COLUMNS:
['PermitNumber', 'PermitNumberCreatedDate', 'IssueDate', 'PermitElapsedDays', 'ProjectValue', 'TypeOfWork', 'Address', 'ProjectDescription', 'PermitCategory', 'Applicant', 'ApplicantAddress', 'PropertyUse', 'SpecificUseCategory', 'BuildingContractor', 'BuildingContractorAddress', 'IssueYear', 'GeoLocalArea', 'Geom', 'YearMonth', 'geo_point_2d']


## 9. Data Types

`pandas` assigns a `dtype` to each column when loading. The main types to watch for:

| dtype | Meaning |
|---|---|
| `int64` | Integer number |
| `float64` | Decimal number |
| `object` | String or mixed content |
| `bool` | True / False |

Columns you expect to be numeric but show `object` likely contain formatting characters (like `$` or `,`) and will need cleaning before any arithmetic can be done on them.

In [9]:
print('=== Property Tax — Data Types ===')
print(prop_sample.dtypes.to_string())

print()

print('=== Permits — Data Types ===')
print(permits_sample.dtypes.to_string())

=== Property Tax — Data Types ===
PID                               str
LEGAL_TYPE                        str
FOLIO                           int64
LAND_COORDINATE                 int64
ZONING_DISTRICT                   str
ZONING_CLASSIFICATION             str
LOT                               str
PLAN                              str
BLOCK                             str
DISTRICT_LOT                      str
FROM_CIVIC_NUMBER                 str
TO_CIVIC_NUMBER               float64
STREET_NAME                       str
PROPERTY_POSTAL_CODE              str
NARRATIVE_LEGAL_LINE1             str
NARRATIVE_LEGAL_LINE2             str
NARRATIVE_LEGAL_LINE3             str
NARRATIVE_LEGAL_LINE4             str
NARRATIVE_LEGAL_LINE5             str
CURRENT_LAND_VALUE            float64
CURRENT_IMPROVEMENT_VALUE     float64
TAX_ASSESSMENT_YEAR           float64
PREVIOUS_LAND_VALUE           float64
PREVIOUS_IMPROVEMENT_VALUE    float64
YEAR_BUILT                    float64
BIG_IMPROVEMENT_

> **What to look for:**
> - **Value columns** (assessed value, permit value) should ideally be `float64` or `int64`. If they show `object`, they contain non-numeric characters — plan to strip and convert during cleaning.
> - **Date columns** will almost always load as `object`. These will need `pd.to_datetime()` conversion in the cleaning step.
> - **ID or code columns** may load as `int64` even if they shouldn't be treated as numbers (e.g., a parcel ID that starts with leading zeros). Flag these — they may need to be cast to `str`.

## 10. Missing Values (Sample)

We count null values per column across the 1,000-row sample. This is not a definitive view of the full dataset, but it gives an early signal of which fields have data quality gaps.

**Important:** `isnull()` only catches `NaN` / `None`. It will *not* catch empty strings, zeros used as placeholders, or text values like `N/A` — those require manual inspection of the raw values.

In [10]:
def missing_summary(df, label):
    '''Print a summary of columns with missing values, sorted by count descending.'''
    missing = df.isnull().sum()
    pct     = (missing / len(df) * 100).round(1)
    summary = pd.DataFrame({'missing_count': missing, 'missing_pct': pct})
    summary = summary[summary['missing_count'] > 0].sort_values('missing_count', ascending=False)

    if summary.empty:
        print(f'{label}: no missing values detected in sample ({len(df)} rows)')
    else:
        print(f'{label} — columns with missing values ({len(df)} rows sampled):')
        display(summary)
    print()

missing_summary(prop_sample,    'Property Tax')
missing_summary(permits_sample, 'Permits')

Property Tax — columns with missing values (1000 rows sampled):


,missing_count,missing_pct
NOTE,1000,100.0
NARRATIVE_LEGAL_LINE5,687,68.7
NARRATIVE_LEGAL_LINE4,663,66.3
BLOCK,610,61.0
FROM_CIVIC_NUMBER,471,47.1
NARRATIVE_LEGAL_LINE3,302,30.2
YEAR_BUILT,45,4.5
BIG_IMPROVEMENT_YEAR,45,4.5
PREVIOUS_IMPROVEMENT_VALUE,28,2.8
PREVIOUS_LAND_VALUE,28,2.8



Permits — columns with missing values (1000 rows sampled):


,missing_count,missing_pct
BuildingContractorAddress,589,58.9
BuildingContractor,433,43.3
PermitCategory,429,42.9
GeoLocalArea,4,0.4
Geom,4,0.4
geo_point_2d,4,0.4
Address,3,0.3
ApplicantAddress,2,0.2


> **What to look for:**
> - Columns with a high missing rate (>50%) may be optional fields, sparsely populated categories, or legacy columns — investigate before deciding to drop them.
> - Missing values in **critical fields** (assessed value, permit date, address) will need an explicit resolution strategy in the cleaning notebook. Do not drop rows yet.
> - If critical columns show 0% missing, check whether zeros or placeholder strings are masking the true null rate — `df['column'].value_counts(dropna=False)` is useful for this.

## 11. Initial Questions to Investigate

Questions raised by this first inspection. These drive the cleaning and analysis work in subsequent notebooks. Move answered questions to `docs/methodology.md`.

### Property Tax Dataset
- What years does this dataset cover? Is there a tax year or assessment year column?
- Is each row one property for one year, or does one row span multiple years?
- What is the unique identifier for a property — is there a parcel ID (PID)?
- Are assessed values already numeric, or do they need parsing (e.g., currency symbols or commas)?
- Does the dataset include multiple property classes (residential, commercial, industrial)? Do we need to filter to residential only?

### Issued Building Permits Dataset
- What date range does this dataset cover — does it span 2019–2024 as expected?
- Is there a field that distinguishes new construction from renovations or demolitions?
- What geographic granularity is available — full address, neighbourhood, local planning area?
- Is there a field for the number of dwelling units permitted?
- Can permits be linked to the property tax dataset by address or parcel identifier?

### Cross-Dataset
- Do the two datasets share a geographic key (address, parcel ID, neighbourhood name) that could support a join?
- Are there time-alignment considerations — e.g., a permit issued in year X may not appear in the assessment roll until year X+1 or X+2?

## Initial Data Understanding Observations

### Property Tax dataset

The Property Tax sample contains 1,000 rows and 30 columns. The dataset appears to be parsed correctly using a semicolon delimiter. Key numeric fields such as `CURRENT_LAND_VALUE`, `CURRENT_IMPROVEMENT_VALUE`, `PREVIOUS_LAND_VALUE`, `PREVIOUS_IMPROVEMENT_VALUE`, and `TAX_LEVY` were read as numeric values, which supports future value calculations.

The most important fields for the property value analysis have relatively low missingness in the sample. Columns with high missingness, such as `NOTE` and some `NARRATIVE_LEGAL_LINE` fields, are not central to the first version of the analysis.

### Building Permits dataset

The Building Permits sample contains 1,000 rows and 20 columns. The dataset also appears to be parsed correctly using a semicolon delimiter. Key fields such as `IssueDate`, `IssueYear`, `YearMonth`, `ProjectValue`, `TypeOfWork`, `PropertyUse`, `SpecificUseCategory`, and `GeoLocalArea` are available for the housing supply signal analysis.

The main missing values in the sample appear in contractor-related fields and `PermitCategory`. These fields are not critical for the initial time-trend analysis. The low missingness in `GeoLocalArea` suggests that area-level permit analysis may be feasible.

### Initial conclusion

Both datasets are suitable for the first version of this portfolio project. The next phase should focus on defining useful derived variables, such as total assessed value, previous total assessed value, year-over-year assessed value change, permit counts by year, and permit project value by year.

---

*Update these questions as analysis progresses. Move resolved questions to `docs/methodology.md`.*